# MAJ-Debate: Run All Proposal Experiments

This notebook aggregates real ablation runs, real DDO labels, real human-eval results, and real external-baseline result files. Synthetic labels and approximate baseline values have been removed.


## 0. Imports & Configuration


In [ ]:
import json, statistics
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists()), cwd)
OUT_DIR = PROJECT_ROOT / 'outputs' / 'experiments'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_MANIFEST = PROJECT_ROOT / 'configs' / 'ablation_manifest.json'
BASELINE_MANIFEST = PROJECT_ROOT / 'configs' / 'external_baselines.json'
if not ABLATION_MANIFEST.exists():
    raise FileNotFoundError(f'Missing real ablation manifest: {ABLATION_MANIFEST}. Copy configs/ablation_manifest.template.json and fill in actual run paths.')
if not BASELINE_MANIFEST.exists():
    raise FileNotFoundError(f'Missing external baseline manifest: {BASELINE_MANIFEST}. Copy configs/external_baselines.template.json and fill in actual result paths.')

with open(ABLATION_MANIFEST, 'r', encoding='utf-8') as f:
    ablation_manifest = json.load(f)
with open(BASELINE_MANIFEST, 'r', encoding='utf-8') as f:
    baseline_manifest = json.load(f)


## 1. Load Helpers and Metric Functions


In [ ]:
def load_json(path_like):
    path = Path(path_like)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if not path.exists():
        raise FileNotFoundError(f'Missing required results file: {path}')
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def evaluate_ddo(stage4_doc):
    rows = []
    for topic in stage4_doc['topics']:
        label = topic.get('benchmark_label')
        if label not in {'PRO', 'CON'}:
            raise ValueError(f"Missing real benchmark_label for DDO topic {topic['topic_id']}")
        rows.append({'topic_id': topic['topic_id'], 'winner': topic['graph_winner'], 'label': label, 'match': int(topic['graph_winner'] == label), 'persuasiveness': topic['persuasiveness'], 'hallucination_conflict': int(topic['hallucination_conflict'])})
    df = pd.DataFrame(rows)
    return df, {'persuasion_agreement': round(df['match'].mean(), 3), 'persuasiveness': round(df['persuasiveness'].mean(), 3), 'judge_conflicts': int(df['hallucination_conflict'].sum())}

def evaluate_human(stage4_doc, human_eval_doc):
    human_topics = human_eval_doc['topic_results']
    rows = []
    for topic in stage4_doc['topics']:
        if topic['topic_id'] not in human_topics:
            raise ValueError(f"Human evaluation result missing for topic {topic['topic_id']}")
        label = human_topics[topic['topic_id']]['majority_label']
        if label not in {'PRO', 'CON'}:
            continue
        rows.append({'topic_id': topic['topic_id'], 'winner': topic['graph_winner'], 'label': label, 'match': int(topic['graph_winner'] == label), 'persuasiveness': topic['persuasiveness']})
    df = pd.DataFrame(rows)
    return df, {'correctness_agreement': round(df['match'].mean(), 3) if not df.empty else None, 'human_persuasiveness': round(df['persuasiveness'].mean(), 3) if not df.empty else None}

def average_attack_diversity(stage2_doc):
    vals = []
    for topic in stage2_doc['topics']:
        premises = {rel.get('attacked_premise') for rel in topic['relations'] if rel.get('kept') and rel['label'] == 'Attack' and rel.get('attacked_premise')}
        vals.append(len(premises))
    return round(statistics.mean(vals), 3) if vals else 0.0

def average_graph_stability(stage3_doc):
    vals = [topic['summary']['graph_stability'] for topic in stage3_doc['topics']]
    return round(statistics.mean(vals), 3) if vals else 0.0


## 2. Aggregate Real Ablation Runs


In [ ]:
ablation_rows = []
error_rows = []
for run in ablation_manifest['runs']:
    stage2_doc = load_json(run['ddo_stage2_path'])
    stage3_doc = load_json(run['ddo_stage3_path'])
    ddo_stage4_doc = load_json(run['ddo_stage4_path'])
    human_stage4_doc = load_json(run['human_stage4_path']) if run.get('human_stage4_path') else None
    human_eval_doc = load_json(run['human_eval_results_path']) if run.get('human_eval_results_path') else None

    ddo_df, ddo_metrics = evaluate_ddo(ddo_stage4_doc)
    human_metrics = {'correctness_agreement': None, 'human_persuasiveness': None}
    if human_stage4_doc and human_eval_doc:
        _, human_metrics = evaluate_human(human_stage4_doc, human_eval_doc)

    row = {
        'configuration': run['configuration'],
        'persuasion_agreement': ddo_metrics['persuasion_agreement'],
        'correctness_agreement': human_metrics['correctness_agreement'],
        'persuasiveness': ddo_metrics['persuasiveness'],
        'attack_diversity': average_attack_diversity(stage2_doc),
        'graph_stability': average_graph_stability(stage3_doc),
        'regret_ddo': round(1.0 - ddo_metrics['persuasion_agreement'], 3),
        'regret_human': round(1.0 - human_metrics['correctness_agreement'], 3) if human_metrics['correctness_agreement'] is not None else None,
    }
    ablation_rows.append(row)
    error_rows.append({'configuration': run['configuration'], 'false_positive_attack_labels': int(sum(1 for topic in stage2_doc['topics'] for rel in topic['relations'] if rel['label'] == 'Attack' and not rel['kept'])), 'fallback_activations': int(sum(1 for topic in stage3_doc['topics'] if topic.get('fallback_reason'))), 'judge_conflicts': int(sum(1 for topic in ddo_stage4_doc['topics'] if topic['hallucination_conflict']))})

ablation_df = pd.DataFrame(ablation_rows)
error_df = pd.DataFrame(error_rows)
display(ablation_df)


## 3. Load Real External Baseline Results


In [ ]:
baseline_rows = []
for item in baseline_manifest['baselines']:
    doc = load_json(item['results_path'])
    baseline_rows.append({'system': doc['system'], 'persuasion_agreement': doc['persuasion_agreement'], 'correctness_agreement': doc.get('correctness_agreement'), 'persuasiveness': doc.get('persuasiveness'), 'attack_diversity': doc.get('attack_diversity'), 'graph_stability': doc.get('graph_stability'), 'notes': doc.get('notes')})
baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)


## 4. Export Final Experiment Artifacts


In [ ]:
comparison_df = pd.concat([ablation_df.rename(columns={'configuration': 'system'}), baseline_df], ignore_index=True, sort=False)
error_analysis = {
    'false_positive_attack_labels': int(error_df['false_positive_attack_labels'].sum()),
    'fallback_activations': int(error_df['fallback_activations'].sum()),
    'judge_conflicts': int(error_df['judge_conflicts'].sum())
}
ablation_out = OUT_DIR / 'ablation_results.json'
baseline_out = OUT_DIR / 'baseline_results.json'
comparison_out = OUT_DIR / 'comparison_tables.csv'
error_out = OUT_DIR / 'error_analysis.json'
with open(ablation_out, 'w', encoding='utf-8') as f:
    json.dump(ablation_rows, f, indent=2)
with open(baseline_out, 'w', encoding='utf-8') as f:
    json.dump(baseline_rows, f, indent=2)
comparison_df.to_csv(comparison_out, index=False)
with open(error_out, 'w', encoding='utf-8') as f:
    json.dump(error_analysis, f, indent=2)
print('Saved experiment artifacts:')
print('-', ablation_out)
print('-', baseline_out)
print('-', comparison_out)
print('-', error_out)
print(json.dumps(error_analysis, indent=2))
